# Modelado CV

* Ines Gonzales
* Jose Arenas
* Jhonier Correa
  
En este punto los datos se encuentran imputados y smoteados.

# Librerias

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn import metrics

# Modelos a evaluar
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier


# Lectura de datos y correccion de tipos

In [2]:
data = pd.read_csv(r'datos\datos_smoteados.csv', index_col=0)
print(data.shape)
data.head()

(9688, 40)


,payer_type,patient_age,breast_cancer_diagnosis_code,population,density,age_median,age_60s,age_70s,married,never_married,...,race_white,race_other,race_multiple,hispanic,disabled,poverty,limited_english,veteran,Ozone,tipo_metastasis
0,COMMERCIAL,62,C50.2_C50.4,39121.878790,2295.939394,38.200000,10.248485,5.951515,50.245455,35.290909,...,54.030303,11.645455,10.081818,37.948485,8.957576,10.109091,8.057576,4.103030,42.301121,Linfatico
1,COMMERCIAL,43,C50.1,21996.683330,626.236667,37.906667,11.426667,6.565000,55.753333,27.195000,...,75.820000,3.816667,6.898333,19.370000,11.253333,9.663333,3.356667,7.446667,40.108207,Linfatico
2,COMMERCIAL,45,C50.2_C50.4,32795.325580,1896.220930,42.871429,12.280952,8.216667,52.604762,31.142857,...,65.014286,5.428571,8.611905,16.716667,8.845238,8.688095,5.280952,4.809524,42.070075,Linfatico
3,'MEDICARE ADVANTAGE',66,C50.9_C79.81,5643.771429,219.362857,45.180000,15.685714,9.445714,51.322857,30.831429,...,93.528571,1.000000,3.262857,2.465714,13.717143,8.888235,0.638235,7.745714,40.107248,VisceralSistemico
4,COMMERCIAL,60,C50.3_C50.5,3404.100000,25.733333,42.790000,14.043333,8.526667,55.886667,24.526667,...,96.406667,0.406667,1.913333,2.723333,15.260000,10.890000,0.503333,7.506667,37.646770,Linfatico


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 9688 entries, 0 to 9687
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   payer_type                    9688 non-null   str    
 1   patient_age                   9688 non-null   int64  
 2   breast_cancer_diagnosis_code  9688 non-null   str    
 3   population                    9688 non-null   float64
 4   density                       9688 non-null   float64
 5   age_median                    9688 non-null   float64
 6   age_60s                       9688 non-null   float64
 7   age_70s                       9688 non-null   float64
 8   married                       9688 non-null   float64
 9   never_married                 9688 non-null   float64
 10  family_size                   9688 non-null   float64
 11  income_household_median       9688 non-null   float64
 12  income_household_10_to_15     9688 non-null   float64
 13  income_househo

In [4]:
data['payer_type'] = data['payer_type'].astype('category')
data['breast_cancer_diagnosis_code'] = data['breast_cancer_diagnosis_code'].astype('category')
data['tipo_metastasis'] = data['tipo_metastasis'].astype('category')

# Division y creacion de pipelines

In [5]:
# 1. Separar características (X) y objetivo (y)
X = data.drop(columns=['tipo_metastasis'])
y = data['tipo_metastasis']

# Codificar la variable objetivo (y) si es categórica
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Identificar columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

# Pipeline para variables numéricas (escalado)
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

# Pipeline para variables categóricas (imputación + One-Hot Encoding)
categorical_transformer = Pipeline(steps=[('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

# Preprocesador global combinando ambos tipos de columnas
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Modelado con cross validation

In [6]:
# 2. Configuración de Cross-Validation y métricas de evaluación
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Definición del diccionario de scoring multiclase (One-vs-Rest Macro)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'f1': 'f1_macro',
    'roc_auc': 'roc_auc_ovr'
}

# Diccionario de modelos a evaluar
modelos = {
    'Random Forest 200': RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    criterion='gini',
    class_weight='balanced',
    random_state=42),

    'Random Forest 400': RandomForestClassifier(
        n_estimators=400,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        criterion='gini',
        class_weight='balanced',
        random_state=42),
    
    'k-NN K=4': KNeighborsClassifier(n_neighbors=4),

    'k-NN K=10': KNeighborsClassifier(n_neighbors=10)
}

# DataFrame para almacenar los promedios finales (filas: métricas, columnas: modelos)
df_promedios_modelos = pd.DataFrame()

# 3. Iteración sobre modelos
for nombre_modelo, algoritmo in modelos.items():
    # Ensamble del pipeline completo
    pipeline_completo = Pipeline(steps=[
        ('preprocesador', preprocessor),
        ('modelo', algoritmo)
    ])
    
    # Ejecución de la validación cruzada
    scores = cross_validate(
        pipeline_completo,
        X,
        y_encoded,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        return_estimator=False
    )
    
    # Construcción de la tabla detallada por iteración (10 Folds)
    df_folds = pd.DataFrame({
        'Fold': [f'Fold_{i+1}' for i in range(cv.n_splits)],
        'Fit_Time': scores['fit_time'],
        'Score_Time': scores['score_time'],
        'Test_Accuracy': scores['test_accuracy'],
        'Test_Precision': scores['test_precision'],
        'Test_Recall': scores['test_recall'],
        'Test_F1': scores['test_f1'],
        'Test_ROC_AUC': scores['test_roc_auc'],
        'Train_Accuracy': scores['train_accuracy'],
        'Train_F1': scores['train_f1']
    }).set_index('Fold')
    
    # Imprimir la tabla específica de este modelo
    print(f"{'='*20} MÉTRICAS POR ITERACIÓN (10-FOLDS): {nombre_modelo} {'='*20}")
    print(df_folds.round(4))
    print('\n')
    
    # Calcular promedios de evaluación en prueba (Test) para la tabla acumulativa global
    promedios_test = {
        'Accuracy': np.mean(scores['test_accuracy']),
        'Precision (Macro)': np.mean(scores['test_precision']),
        'Recall (Macro)': np.mean(scores['test_recall']),
        'F1-Score (Macro)': np.mean(scores['test_f1']),
        'ROC-AUC (OvR)': np.mean(scores['test_roc_auc']),
        'Fit Time (s)': np.mean(scores['fit_time'])
    }
    
    # Almacenar en el DataFrame final sin imprimir
    df_promedios_modelos[nombre_modelo] = pd.Series(promedios_test).round(4)

==================== MÉTRICAS POR ITERACIÓN (10-FOLDS): Random Forest 200 ====================
         Fit_Time  Score_Time  Test_Accuracy  Test_Precision  Test_Recall  \
Fold                                                                        
Fold_1     4.9572      0.1686         0.4954          0.4433       0.4589   
Fold_2     4.8354      0.2039         0.4747          0.4228       0.4390   
Fold_3     4.8468      0.2389         0.4943          0.4389       0.4534   
Fold_4     5.0929      0.1490         0.4561          0.4045       0.4160   
Fold_5     3.9363      0.1271         0.4696          0.4166       0.4267   
Fold_6     3.8869      0.1224         0.5005          0.4384       0.4499   
Fold_7     4.0652      0.1843         0.4489          0.4077       0.4201   
Fold_8     4.0571      0.1319         0.5036          0.4458       0.4633   
Fold_9     4.4184      0.1803         0.4959          0.4445       0.4582   
Fold_10    4.1365      0.1624         0.4721          0.42

# Medidas de evaluacion

In [7]:
df_promedios_modelos

,Random Forest 200,Random Forest 400,k-NN K=4,k-NN K=10
Accuracy,0.4811,0.4833,0.5449,0.5746
Precision (Macro),0.4290,0.4285,0.4102,0.4060
Recall (Macro),0.4425,0.4436,0.3973,0.3730
F1-Score (Macro),0.4245,0.4262,0.3944,0.3609
ROC-AUC (OvR),0.6194,0.6106,0.5772,0.5815
Fit Time (s),4.4232,10.7541,0.0346,0.0391


In [10]:
# Lista para acumular los registros del diagnóstico
registros_diagnostico = []

for nombre_modelo, algoritmo in modelos.items():
    pipeline_completo = Pipeline(steps=[
        ('preprocesador', preprocessor),
        ('modelo', algoritmo)
    ])
    
    # Validación cruzada obteniendo puntajes de entrenamiento y prueba
    scores = cross_validate(
        pipeline_completo, 
        X, 
        y_encoded, 
        cv=cv, 
        scoring=scoring, 
        return_train_score=True
    )
    
    # Extraer métricas para cada criterio
    for metrica in scoring.keys():
        train_score = scores[f'train_{metrica}'].mean()
        test_score = scores[f'test_{metrica}'].mean()
        gap = ((train_score - test_score)*100)/train_score
        
        # Evaluar el estado según la brecha
        if train_score < 0.70 and test_score < 0.70:
            estado = 'Underfitting'
        elif gap > 0.10:
            estado = 'Overfitting'
        else:
            estado = 'Buen Ajuste'
            
        registros_diagnostico.append({
            'Modelo': nombre_modelo,
            'Métrica': metrica.upper(),
            'Train (Promedio)': round(train_score, 4),
            'Test (Promedio)': round(test_score, 4),
            'Gap % (Train - Test)': round(gap, 4),
            'Diagnóstico': estado
        })

# Convertir la lista de diccionarios en un DataFrame
df_diagnostico_ajuste = pd.DataFrame(registros_diagnostico)

In [11]:
df_diagnostico_ajuste

,Modelo,Métrica,Train (Promedio),Test (Promedio),Gap % (Train - Test),Diagnóstico
0,Random Forest 200,ACCURACY,0.6450,0.4811,25.4146,Underfitting
1,Random Forest 200,PRECISION,0.6075,0.4290,29.3794,Underfitting
2,Random Forest 200,RECALL,0.6694,0.4425,33.8946,Underfitting
3,Random Forest 200,F1,0.6170,0.4245,31.2037,Underfitting
4,Random Forest 200,ROC_AUC,0.8442,0.6194,26.6370,Overfitting
5,Random Forest 400,ACCURACY,0.7295,0.4833,33.7556,Overfitting
6,Random Forest 400,PRECISION,0.6913,0.4285,38.0172,Underfitting
7,Random Forest 400,RECALL,0.7714,0.4436,42.4958,Overfitting
8,Random Forest 400,F1,0.7081,0.4262,39.8082,Overfitting
9,Random Forest 400,ROC_AUC,0.9122,0.6106,33.0624,Overfitting
